In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    make_scorer,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from utils import custom_score, perform_random_search_cv, get_models, score_with_thresh

SEED = 3105
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# numbers of features to test
K_VALUES = [3, 4, 5, 6, 7]
# number of random feature subsets
N_RANDOM = 3
# number of random hyperparameter configs per each model
N_ITER = 20
# fraction of a set we are allowed to contact
CONTACT_RATE = 0.2

## Prepare data

In [3]:
data_dir = Path("../../data")
X = pd.read_csv(data_dir / "x_train.txt", sep=" ")
y = pd.read_csv(data_dir / "y_train.txt", sep=" ").values.ravel()

with open("../feature_selection/selected_features.txt") as f:
    selected = [s.strip().strip("'").strip('"') for s in f.read().split(",")]

X = X[selected]
print(f"X: {X.shape}")

X: (5000, 30)


In [4]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED
)

X_train.reset_index(drop=True, inplace=True)
X_val.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)

print(f"train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")

train: (3000, 30)  val: (1000, 30)  test: (1000, 30)


## Feature importance and subsets to test

In [5]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
ranking = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(
    ascending=False
)
top15 = ranking.index[:15].tolist()


def random_subsets(k, n, exclude):
    seen = {frozenset(exclude)}
    subsets = []
    while len(subsets) < n:
        s = frozenset(rng.choice(top15, size=k, replace=False))
        if s not in seen:
            seen.add(s)
            subsets.append(sorted(s, key=top15.index))
    return subsets


subsets = {}
for k in K_VALUES:
    topk = ranking.index[:k].tolist()
    topk_1 = ranking.index[1 : k + 1].tolist()
    topk_2 = ranking.index[2 : k + 2].tolist()
    subsets[k] = (
        [("top", topk)]
        + [("top+1", topk_1)]
        + [("top+2", topk_2)]
        + [(f"rand_{i}", s) for i, s in enumerate(random_subsets(k, N_RANDOM, topk))]
    )
    print(f"k={k}: {len(subsets[k])} subsets")

k=3: 6 subsets
k=4: 6 subsets
k=5: 6 subsets
k=6: 6 subsets
k=7: 6 subsets


## Random search CV

In [ ]:
results = perform_random_search_cv(
    X_train, y_train, seed=SEED, k_values=K_VALUES, n_iter=N_ITER, subsets=subsets
)

In [6]:
results = pd.read_csv("cv_results_1780410113_seed_3105.csv")
results.head(16)
# the top 16 models have score spread 580 - 530 = 50 points, the average std is ~36 points. 
# Std is nearly as large as the full ranking range - we cannot choose only based on noisy cv estimates. 
# the models here are very similar.

,k,kind,model,cv_score,cv_score_std,features,best_params,accuracy,accuracy_std,balanced_accuracy,balanced_accuracy_std,precision,precision_std
0,3,top,ExtraTrees,580.0,28.284271,"['V255', 'V191', 'V176']","{'clf__max_depth': 6, 'clf__min_samples_leaf':...",0.611000,0.011314,0.610521,0.011277,0.637447,0.013943
1,3,top+2,XGBoost,570.0,28.284271,"['V176', 'V380', 'V160']",{'clf__colsample_bytree': np.float64(0.7420988...,0.592667,0.009741,0.592259,0.009702,0.610767,0.013374
2,3,top,XGBoost,565.0,30.822070,"['V255', 'V191', 'V176']",{'clf__colsample_bytree': np.float64(0.9781553...,0.615667,0.002357,0.615373,0.002348,0.630537,0.004564
3,3,top,GradientBoosting,565.0,30.822070,"['V255', 'V191', 'V176']",{'clf__learning_rate': np.float64(0.0124474203...,0.614333,0.012284,0.613986,0.012211,0.632045,0.014980
4,3,top,RandomForest,560.0,24.494897,"['V255', 'V191', 'V176']","{'clf__max_depth': 6, 'clf__min_samples_leaf':...",0.617667,0.009741,0.617239,0.009621,0.641139,0.014075
5,3,top,SVM_RBF,560.0,21.213203,"['V255', 'V191', 'V176']","{'clf__C': np.float64(0.23522021535538676), 'c...",0.612333,0.008179,0.611841,0.008041,0.639095,0.014112
6,3,top+2,RandomForest,560.0,36.742346,"['V176', 'V380', 'V160']","{'clf__max_depth': 5, 'clf__min_samples_leaf':...",0.600000,0.007789,0.599522,0.007712,0.623639,0.010810
7,3,top+1,ExtraTrees,555.0,30.822070,"['V191', 'V176', 'V380']","{'clf__max_depth': 10, 'clf__min_samples_leaf'...",0.603333,0.002625,0.602884,0.002652,0.625796,0.004788
8,3,rand_2,ExtraTrees,555.0,83.366660,"[np.str_('V176'), np.str_('V416'), np.str_('V2...","{'clf__max_depth': 9, 'clf__min_samples_leaf':...",0.595000,0.013880,0.594488,0.013806,0.619595,0.017404
9,3,top+2,GradientBoosting,555.0,25.495098,"['V176', 'V380', 'V160']",{'clf__learning_rate': np.float64(0.0124474203...,0.600667,0.010077,0.600262,0.010033,0.619405,0.011353


## Choosing threshold on val and evaluating on different val/test splits

In [7]:
top16_models = results.head(16)
models = get_models()

In [8]:
seeds = [(2910 + i) for i in range(30)]
ts = np.linspace(0, 0.9, num=30)
per_split = {i: [] for i in range(16)}

X_valte = pd.concat([X_val, X_test], axis=0)
y_valte = pd.concat([pd.DataFrame(y_val), pd.DataFrame(y_test)], axis=0)

for i in range(16):
    k = top16_models["k"].iloc[i]
    features = eval(top16_models["features"].iloc[i])
    params = eval(top16_models["best_params"].iloc[i])
    pipeline = Pipeline([("scaler", StandardScaler()),
                         ("clf", clone(models[top16_models["model"].iloc[i]][0]))])
    pipeline.set_params(**params)
    pipeline.fit(X_train[features], y_train)

    for s in seeds:
        X_val_s, X_test_s, y_val_s, y_test_s = train_test_split(
            X_valte, y_valte, test_size=0.5, random_state=s
        )
        y_val_s = y_val_s.to_numpy().ravel()
        y_test_s = y_test_s.to_numpy().ravel()

        proba_val = pipeline.predict_proba(X_val_s[features])[:, 1]
        best_t, best_val = 0.0, -np.inf
        for t in ts:
            score, _ = score_with_thresh(y_val_s, proba_val, n_var=k, thresh=t)
            if score >= best_val:
                best_val, best_t = score, t

        proba_te = pipeline.predict_proba(X_test_s[features])[:, 1]
        test_sc, test_contacts = score_with_thresh(y_test_s, proba_te, n_var=k, thresh=best_t)

        per_split[i].append({"seed": s, "val_thresh": best_t,
                             "val_score": best_val, "test_score": test_sc,
                             "contacts": test_contacts})

In [9]:
rows = []
for i in range(16):
    d = pd.DataFrame(per_split[i])
    name = f"{top16_models['model'].iloc[i]} {i + 1}"
    thresh_mean, thresh_std = d["val_thresh"].mean(), d["val_thresh"].std(ddof=1)
    vmean, vstd = d["val_score"].mean(), d["val_score"].std(ddof=1)
    tmean, tstd = d["test_score"].mean(), d["test_score"].std(ddof=1)
    rows.append({
        "model": name,
        "k": int(top16_models["k"].iloc[i]),
        "cv": top16_models["cv_score"].iloc[i],
        "mean_val_thresh": round(thresh_mean, 4),
        "std_val_thresh": round(thresh_std, 3),
        "val_mean": round(vmean, 1),
        "val_std": round(vstd, 1),
        "test_mean": round(tmean, 1),
        "test_std": round(tstd, 1),
        "test_min": d["test_score"].min(),
        "test_max": d["test_score"].max(),
        "criterion": round(tmean - tstd, 1),
        "med_contacts": int(d["contacts"].median()),
    })

summary = pd.DataFrame(rows).sort_values("criterion", ascending=False)
summary

,model,k,cv,mean_val_thresh,std_val_thresh,val_mean,val_std,test_mean,test_std,test_min,test_max,criterion,med_contacts
0,ExtraTrees 1,3,580.0,0.5224,0.012,602.5,61.2,621.3,58.7,500,740,562.7,200
13,SVM_RBF 14,3,530.0,0.5855,0.011,568.0,72.6,588.3,70.4,470,725,517.9,200
10,SVM_RBF 11,3,550.0,0.6186,0.008,554.5,64.5,574.3,66.9,430,695,507.4,200
7,ExtraTrees 8,3,555.0,0.5214,0.013,562.0,78.4,582.8,82.3,415,755,500.6,200
11,SVM_RBF 12,3,535.0,0.6155,0.012,555.0,67.2,562.7,68.3,445,695,494.4,200
3,GradientBoosting 4,3,565.0,0.5834,0.013,541.0,73.1,553.7,67.1,405,680,486.6,200
5,SVM_RBF 6,3,560.0,0.6248,0.011,552.5,65.0,552.3,79.4,345,680,472.9,200
12,RandomForest 13,3,530.0,0.5514,0.013,540.5,96.0,559.0,102.5,375,770,456.5,200
15,ExtraTrees 16,3,530.0,0.4966,0.000,527.5,72.1,526.0,74.4,365,665,451.6,200
4,RandomForest 5,3,560.0,0.5741,0.016,500.0,70.4,508.7,73.8,345,650,434.9,200


# Final model and test prediction

In [10]:
data_dir = Path("../../data")
X_train = pd.read_csv(data_dir / "x_train.txt", sep=" ")
y_train = pd.read_csv(data_dir / "y_train.txt", sep=" ").values.ravel()
X_test = pd.read_csv(data_dir / "x_test.txt", sep=" ")

In [11]:
print("Final model - Extra Trees")
features = eval(top16_models["features"].iloc[0])
print(f"Chosen features: {features}")
params = eval(top16_models["best_params"].iloc[0])
print(f"Parameters: {params}")
et = ExtraTreesClassifier(max_depth=6, min_samples_leaf=2, n_estimators=238)
pipeline = Pipeline([("scaler", StandardScaler()), ("clf", et)])
X_train = X_train[features]
X_test = X_test[features]
pipeline.fit(X_train, y_train)

Final model - Extra Trees
Chosen features: ['V255', 'V191', 'V176']
Parameters: {'clf__max_depth': 6, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 238}


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",238
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


### Prediction

In [12]:
pred = pipeline.predict_proba(X_test)[:,1]
pred_thresh= (pred >= (0.5224))
print(f"number of contacted clients with validation threshold: {sum(pred_thresh)}")

number of contacted clients with validation threshold: 1221


In [15]:
order = np.argsort(-pred)
p_sorted = pred[order]
print("probability at rank 1000:", round(p_sorted[999], 4))

top = 1000
# indices of top 1000 probabilities
top_indices = np.argsort(pred)[-top:] + 1

probability at rank 1000: 0.531


In [16]:
top_indices

array([4731, 2714, 1897, 2123, 3096,  524, 3500, 3412, 2688,  215, 3025,
       4219, 3108, 4025,  235, 4094, 2535, 1079, 1801, 4207, 2589, 4533,
       4476, 4960,  277, 4103, 3395, 4374, 3656,  406,  985, 2298, 3389,
       3849, 4424, 3518, 3618, 2927, 1772, 2085,  263, 1785, 2205, 3062,
       1544, 4078, 1587, 1191, 4940, 4345, 1482, 3680, 3259, 2923, 1223,
       1989, 1713,    8, 2531,  501, 2822,  874, 3356, 1985,  398, 4799,
       1340, 4645, 4845,  247, 4853,  658, 1398, 2612, 1043, 4339, 3740,
        289, 3703,  445, 1503, 3459, 4920, 1566, 1829, 2558, 1877,  339,
       3144, 3285, 2566,  376, 1886,  966, 3598, 1978, 3353, 1522,  264,
       1092, 3657, 2763, 1520, 1747, 1008, 2640,    1, 2991, 2220, 3629,
       3193, 3103, 4190,  733, 3313, 2876, 4914, 4444, 1673, 4104,  268,
        460, 3206,  180, 2449, 2235, 2044,  456, 1355, 1262, 3718, 4028,
        671, 3117, 4808, 3247, 4448, 3057, 1611, 2018, 2851, 4337, 2895,
       1135, 3580, 2950, 3277, 3797, 3993, 3077, 26